# Standalone NCF Model Notebook

This notebook is self-contained. It documents the NCF architecture, implicit-feedback training loop, and the cold-start fix where a new user's embedding is initialized from an SBERT profile vector instead of zeros or random noise.

## Feature Stream

The recommender does not use a static domain map. Scraped job text is embedded by SBERT and used to initialize item factors; user preference comes from implicit feedback.

In [1]:
import hashlib
import math

import numpy as np
import torch
from torch import nn

DIM = 16
torch.manual_seed(42)


def seeded_vector(seed, dim=DIM):
    raw = np.frombuffer(
        hashlib.blake2b(seed.encode('utf-8'), digest_size=32).digest(),
        dtype=np.uint8,
    ).astype(np.float32)
    values = np.tile(raw, math.ceil(dim / raw.size))[:dim]
    values = values / 127.5 - 1.0
    norm = np.linalg.norm(values)
    return values / norm if norm else values


def sigmoid(value):
    return float(1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value)))))


## Architecture

The served NCF boundary is NeuMF: a GMF branch, an MLP branch, and a fusion head. Stable string IDs map users and jobs to embedding rows, while profile/job vectors remain useful for cold-start feature initialization.


In [2]:
class TinyNeuMF(nn.Module):
    def __init__(self, max_users=16, max_items=16, dim=DIM, hidden=32):
        super().__init__()
        self.user_gmf = nn.Embedding(max_users, dim)
        self.item_gmf = nn.Embedding(max_items, dim)
        self.user_mlp = nn.Embedding(max_users, dim)
        self.item_mlp = nn.Embedding(max_items, dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim * 2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
        )
        self.out = nn.Linear(dim + hidden // 2, 1)

    def forward(self, user_ids, item_ids):
        gmf = self.user_gmf(user_ids) * self.item_gmf(item_ids)
        mlp = self.mlp(torch.cat([self.user_mlp(user_ids), self.item_mlp(item_ids)], dim=-1))
        return self.out(torch.cat([gmf, mlp], dim=-1)).squeeze(-1)


model = TinyNeuMF()
user_index = {}
item_index = {}


def stable_index(mapping, key):
    key = str(key)
    if key not in mapping:
        mapping[key] = len(mapping)
    return mapping[key]


def ncf_predict(user_id, job_id):
    user_tensor = torch.tensor([stable_index(user_index, user_id)], dtype=torch.long)
    item_tensor = torch.tensor([stable_index(item_index, job_id)], dtype=torch.long)
    with torch.inference_mode():
        return sigmoid(float(model(user_tensor, item_tensor).item()))


print(ncf_predict('u-1', 'mc'))


0.4010817118308619


## Implicit Feedback Training Loop

Clicks, applications, and saves are positive signals; skips and impressions are weak or negative signals. This notebook uses `BCEWithLogitsLoss` on explicit user-item rows so the artifact mirrors the service's NeuMF training contract.


In [3]:
implicit_feedback = [
    {'user_id': 'u-1', 'job_id': 'mc', 'label': 1.0},
    {'user_id': 'u-1', 'job_id': 'translator', 'label': 1.0},
    {'user_id': 'u-1', 'job_id': 'backend', 'label': 0.0},
    {'user_id': 'u-2', 'job_id': 'backend', 'label': 1.0},
    {'user_id': 'u-2', 'job_id': 'mc', 'label': 0.0},
]

optimizer = torch.optim.AdamW(model.parameters(), lr=0.02, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()
losses = []

for _ in range(30):
    for row in implicit_feedback:
        users = torch.tensor([stable_index(user_index, row['user_id'])], dtype=torch.long)
        items = torch.tensor([stable_index(item_index, row['job_id'])], dtype=torch.long)
        labels = torch.tensor([row['label']], dtype=torch.float32)
        loss = loss_fn(model(users, items), labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(float(loss.item()))

print('mean_loss', round(float(np.mean(losses)), 4))


mean_loss 0.0496


## Cold Start Evaluation

A new user with no history still receives varied scores because stable mappings and initialized embeddings produce non-constant NeuMF logits. Production service blends this NeuMF path with cold-start factor features from profile/job vectors.


In [4]:
scores = [ncf_predict(user_id='u-new', job_id=f'job-{j}') for j in range(10)]
print([round(score, 4) for score in scores])
assert len(set(round(score, 2) for score in scores)) > 1, 'NCF must NOT output constant 0.5'
assert any(abs(score - 0.5) > 0.01 for score in scores), 'Cold-start scores must vary based on NeuMF embeddings'


[0.9952, 0.9628, 0.9868, 0.2005, 0.953, 0.9989, 0.9733, 0.9988, 0.9319, 0.7727]
